# Andrew Fox - Defender Hydrodynamic Modeling ML Algorithm Using Off-Diagonal Terms

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import sklearn
from sklearn.model_selection import train_test_split
import torch.optim as optim
import torch.nn.functional as F


print("Library Versions:")
print('numpy:',np.__version__)
print('pandas:',pd.__version__)
print('torch:',torch.__version__)
print('sklearn:',sklearn.__version__)
print(torch.cuda.is_available())          # Should print: True
print(torch.cuda.get_device_name(0))      # Should print: your GPU model


Library Versions:
numpy: 1.23.5
pandas: 2.3.2
torch: 2.8.0+cu128
sklearn: 1.7.2
True
NVIDIA GeForce RTX 5070 Ti Laptop GPU


# Import Simulation/Experimental Data

In [2]:
# Read the dataset
rov = pd.read_csv(
    "/home/andrew/fossen_ml_pipeline/defender_parameter_estimator/csv_files/Coupled Maneuvers/defender_data_teleop_circle.csv",
    sep="\t"
)

# Drop the first 3 rows
rov = rov.iloc[3:].reset_index(drop=True)

# -------------------------------
# Columns to require (no NaNs)
# -------------------------------
required_signal_cols = [
    "u_dot","v_dot","w_dot","p_dot","q_dot","r_dot",
    "u","v","w","p","q","r",
    "x","y","z","phi","theta","psi",
    "X","Y","Z","K","M","N",
]

# Warn if any required columns are missing
missing_signal_cols = [c for c in required_signal_cols if c not in rov.columns]

if missing_signal_cols:
    print("WARNING: Missing required signal columns:")
    for c in missing_signal_cols:
        print(f"  - {c}")

# Keep only columns that actually exist, so later code does not crash
signal_cols = [c for c in required_signal_cols if c in rov.columns]

# Optional: print what will actually be used
print(f"Using {len(signal_cols)} signal columns:")
print(signal_cols)

# If you want to ALSO require the valid flags (optional)
flag_cols = [c for c in ["pose_valid", "twist_valid", "wrench_valid"] if c in rov.columns]
age_cols  = [c for c in ["pose_age", "twist_age", "wrench_age"] if c in rov.columns]

# Make flags ints (in case they come in as floats)
for c in flag_cols:
    rov[c] = rov[c].fillna(0).astype(int)

# -------------------------------
# 1) DROP ROWS WITH NaNs
# -------------------------------
before = len(rov)
rov = rov.dropna(subset=signal_cols).reset_index(drop=True)
after = len(rov)
print(f"Dropped {before - after} rows due to NaNs in required signal columns.")

# -------------------------------
# 2) Epsilon-zero + rounding
# -------------------------------
epsilon = 1e-4

arr = rov[signal_cols].to_numpy(dtype=float)

# No NaNs should remain, but keep this safe anyway
finite = np.isfinite(arr)
arr[finite & (np.abs(arr) < epsilon)] = 0.0
arr[finite] = np.round(arr[finite], 5)

rov[signal_cols] = arr

# Keep ages tidy (optional)
for c in age_cols:
    rov[c] = rov[c].astype(float).round(6)

# Print cleaned result
print("Cleaned shape:", rov.shape)
print("First few rows:\n", rov.head())

# Sanity check: should be 0 now for required signal columns
print("\nNaNs per column (top 10):")
print(rov.isna().sum().sort_values(ascending=False).head(10))

Using 24 signal columns:
['u_dot', 'v_dot', 'w_dot', 'p_dot', 'q_dot', 'r_dot', 'u', 'v', 'w', 'p', 'q', 'r', 'x', 'y', 'z', 'phi', 'theta', 'psi', 'X', 'Y', 'Z', 'K', 'M', 'N']
Dropped 0 rows due to NaNs in required signal columns.
Cleaned shape: (18093, 25)
First few rows:
            time    u_dot    v_dot    w_dot  p_dot  q_dot  r_dot        u  \
0  1.764881e+09 -0.05917 -0.02033 -0.08339    0.0    0.0    0.0  0.03676   
1  1.764881e+09 -0.05037 -0.01794 -0.07969    0.0    0.0    0.0  0.03676   
2  1.764881e+09 -0.05795 -0.01017 -0.07975    0.0    0.0    0.0  0.03676   
3  1.764881e+09 -0.07585 -0.01398 -0.08190    0.0    0.0    0.0  0.03676   
4  1.764881e+09 -0.06125 -0.01433 -0.07056    0.0    0.0    0.0  0.03676   

         v        w  ...        z      phi    theta     psi    X    Y    Z  \
0  0.01695 -0.00875  ... -2.07594 -0.01916 -0.06667 -1.3436  0.0  0.0  0.0   
1  0.01695 -0.00875  ... -2.07594 -0.01916 -0.06667 -1.3436  0.0  0.0  0.0   
2  0.01695 -0.00875  ... -2.0759

In [9]:
# ==== RANK CHECK =====

# Extract surge velocity and acceleration
u_dot = rov["u_dot"].to_numpy()
u     = rov["u"].to_numpy()

# Build surge regressor matrix Phi_X
Phi_X = np.column_stack([
    u_dot,
    u,
    np.abs(u) * u
])  # shape (N, 3)

print("Phi_X shape:", Phi_X.shape)

rank = np.linalg.matrix_rank(Phi_X)
print("Rank of Phi_X:", rank)

U, S, Vt = np.linalg.svd(Phi_X, full_matrices=False)
print("Singular values:", S)
print("Condition number:", S[0] / S[-1])
print("Smallest-singular direction Vt[-1]:", Vt[-1])


Phi_X shape: (68259, 3)
Rank of Phi_X: 3
Singular values: [135.69377815  75.43757547  21.6203745 ]
Condition number: 6.276199246975281
Smallest-singular direction Vt[-1]: [ 0.00273405  0.56671469 -0.82390958]


In [15]:
# ============================================================
# Single-DOF filter: remove rows with inactive DOF commands
# but keep active-DOF coast-down/zero-command data
# ============================================================

active_dof = "X"

force_cols = ["X", "Y", "Z", "K", "M", "N"]
other_dofs = [c for c in force_cols if c != active_dof]

inactive_max = {
    "X": 0.5,
    "Y": 0.5,
    "Z": 0.5,
    "K": 0.05,
    "M": 0.05,
    "N": 0.05,
}

missing_force_cols = [c for c in force_cols if c not in rov.columns]
if missing_force_cols:
    raise ValueError(f"Missing force/moment columns needed for DOF filtering: {missing_force_cols}")

inactive_mask = np.ones(len(rov), dtype=bool)

for c in other_dofs:
    inactive_mask &= rov[c].abs().to_numpy() < inactive_max[c]

before = len(rov)
rov = rov[inactive_mask].reset_index(drop=True)
after = len(rov)

print(f"\nSingle-DOF filter active DOF: {active_dof}")
print("Filter mode: removing inactive-DOF command rows only")
print(f"Original rows: {before}")
print(f"Kept rows:     {after}")
print(f"Removed rows:  {before - after}")

print("\nMean absolute force/moment after filtering:")
print(rov[force_cols].abs().mean())

print("\nMax absolute force/moment after filtering:")
print(rov[force_cols].abs().max())


Single-DOF filter active DOF: X
Filter mode: removing inactive-DOF command rows only
Original rows: 68259
Kept rows:     56302
Removed rows:  11957

Mean absolute force/moment after filtering:
X    19.952646
Y     0.000000
Z     0.000000
K     0.000000
M     0.000000
N     0.000000
dtype: float64

Max absolute force/moment after filtering:
X    77.49032
Y     0.00000
Z     0.00000
K     0.00000
M     0.00000
N     0.00000
dtype: float64


In [3]:
# === Clean and split ROV dataset for PyTorch ===

# Explicit column groups
nu_dot_cols = ["u_dot", "v_dot", "w_dot", "p_dot", "q_dot", "r_dot"]
nu_cols     = ["u", "v", "w", "p", "q", "r"]
eta_cols    = ["x", "y", "z", "phi", "theta", "psi"]
tau_cols    = ["X", "Y", "Z", "K", "M", "N"]

required_model_cols = nu_dot_cols + nu_cols + eta_cols + tau_cols

# Check for missing columns
missing_model_cols = [c for c in required_model_cols if c not in rov.columns]
if missing_model_cols:
    raise ValueError(f"Missing required model columns: {missing_model_cols}")

# Build model dataframe in the exact order we want
rov_model = rov[required_model_cols].copy()

# Convert to NumPy
rov_np = rov_model.to_numpy(dtype=float)

# Train/test split
rov_train_np, rov_test_np = train_test_split(
    rov_np,
    test_size=0.25,
    random_state=42
)

# Split by fixed known structure
nu_dot_train = rov_train_np[:, 0:6]
nu_train     = rov_train_np[:, 6:12]
eta_train    = rov_train_np[:, 12:18]
y_train_np   = rov_train_np[:, 18:24]

nu_dot_test = rov_test_np[:, 0:6]
nu_test     = rov_test_np[:, 6:12]
eta_test    = rov_test_np[:, 12:18]
y_test_np   = rov_test_np[:, 18:24]

# Convert to PyTorch tensors
nu_dot_train = torch.tensor(nu_dot_train, dtype=torch.float32)
nu_train     = torch.tensor(nu_train,     dtype=torch.float32)
eta_train    = torch.tensor(eta_train,    dtype=torch.float32)
y_train      = torch.tensor(y_train_np,   dtype=torch.float32)

nu_dot_test = torch.tensor(nu_dot_test, dtype=torch.float32)
nu_test     = torch.tensor(nu_test,     dtype=torch.float32)
eta_test    = torch.tensor(eta_test,    dtype=torch.float32)
y_test      = torch.tensor(y_test_np,   dtype=torch.float32)

# Optional sanity print
print("Training tensor shapes:")
print("nu_dot_train:", nu_dot_train.shape)
print("nu_train:    ", nu_train.shape)
print("eta_train:   ", eta_train.shape)
print("y_train:     ", y_train.shape)

print("\nTesting tensor shapes:")
print("nu_dot_test:", nu_dot_test.shape)
print("nu_test:    ", nu_test.shape)
print("eta_test:   ", eta_test.shape)
print("y_test:     ", y_test.shape)

print("Mean |nu_dot_train|:", torch.mean(torch.abs(nu_dot_train), dim=0))
print("Mean |nu_train|:    ", torch.mean(torch.abs(nu_train), dim=0))
print("Mean |y_train|:     ", torch.mean(torch.abs(y_train), dim=0))

Training tensor shapes:
nu_dot_train: torch.Size([13569, 6])
nu_train:     torch.Size([13569, 6])
eta_train:    torch.Size([13569, 6])
y_train:      torch.Size([13569, 6])

Testing tensor shapes:
nu_dot_test: torch.Size([4524, 6])
nu_test:     torch.Size([4524, 6])
eta_test:    torch.Size([4524, 6])
y_test:      torch.Size([4524, 6])
Mean |nu_dot_train|: tensor([0.0544, 0.1751, 0.0491, 1.0819, 0.7573, 0.9798])
Mean |nu_train|:     tensor([0.4986, 0.1060, 0.0107, 0.0428, 0.0399, 0.3720])
Mean |y_train|:      tensor([21.9917,  4.8359,  0.1572,  0.0000,  0.0000,  1.1939])


In [10]:

"""
    Fossen-style inverse dynamics model:

        tau = M(nu_dot) + C(nu)nu + D(nu)nu + g(eta)

    with toggles to enable/disable individual components so MLE/MAP/HMC
    all share the exact same implementation.

    Notes:
    - NED convention (z down) as in your current code.
    - Added-mass coefficients are used in M_A always (as you currently do).
    - C_A (added-mass Coriolis) can be toggled off to avoid Munk-moment-induced instability
      when off-diagonal damping is not modeled.
    """

class ROVDynamicsModel(nn.Module):
    def __init__(
        self,
        use_Crb: bool = True,
        use_Ca: bool = True,
        use_D: bool = True,
        use_g: bool = True,
        m_val: float = 23.89, #17.2 for sim and 23.89 for real world

        # NEW:
        learn_inertia: bool = False,
        Ixx_cad: float = 0.393,
        Iyy_cad: float = 1.302,
        Izz_cad: float = 1.429,
    ):
        super().__init__()

        self.use_Crb = use_Crb
        self.use_Ca  = use_Ca
        self.use_D   = use_D
        self.use_g   = use_g

        # --------------------------
        # (1) Added-Mass Coefficients
        # --------------------------
        self.X_dot_u = nn.Parameter(torch.tensor(-0.0))
        self.Y_dot_v = nn.Parameter(torch.tensor(-0.0))
        self.Z_dot_w = nn.Parameter(torch.tensor(-0.0))
        self.K_dot_p = nn.Parameter(torch.tensor(-0.0))
        self.M_dot_q = nn.Parameter(torch.tensor(-0.0))
        self.N_dot_r = nn.Parameter(torch.tensor(-0.0))

        #Sway-Yaw Coupling
        self.Y_dot_r = nn.Parameter(torch.tensor(0.0)) #assuming symmetrical added mass

        # --------------------------
        # (2) Rigid-Body Inertias
        # --------------------------
        self.learn_inertia = bool(learn_inertia)

        # Store CAD inertias as buffers so they move with .to(device)
        # and appear in state_dict, but are NOT trainable params.
        self.register_buffer("Ixx_cad", torch.tensor(float(Ixx_cad), dtype=torch.float32))
        self.register_buffer("Iyy_cad", torch.tensor(float(Iyy_cad), dtype=torch.float32))
        self.register_buffer("Izz_cad", torch.tensor(float(Izz_cad), dtype=torch.float32))

        if self.learn_inertia:
            self.I_xx = nn.Parameter(self.Ixx_cad.clone())
            self.I_yy = nn.Parameter(self.Iyy_cad.clone())
            self.I_zz = nn.Parameter(self.Izz_cad.clone())
        else:
            # Not Parameters → optimizer can’t change them
            self.I_xx = None
            self.I_yy = None
            self.I_zz = None

        # --------------------------
        # (3) Center of Gravity Offsets
        # --------------------------
        self.x_g = nn.Parameter(torch.tensor(0.0))
        self.y_g = nn.Parameter(torch.tensor(0.0))
        self.z_g = nn.Parameter(torch.tensor(0.0))

        # --------------------------
        # (4) Linear Damping
        # --------------------------
        self.X_u = nn.Parameter(torch.tensor(0.0))
        self.Y_v = nn.Parameter(torch.tensor(0.0))
        self.Z_w = nn.Parameter(torch.tensor(0.0))
        self.K_p = nn.Parameter(torch.tensor(0.0))
        self.M_q = nn.Parameter(torch.tensor(0.0))
        self.N_r = nn.Parameter(torch.tensor(0.0))

        #Sway-Yaw Coupling
        self.Y_r = nn.Parameter(torch.tensor(0.0))
        self.N_v = nn.Parameter(torch.tensor(0.0))

        # --------------------------
        # (5) Quadratic Damping
        # --------------------------
        self.X_uu = nn.Parameter(torch.tensor(0.0))
        self.Y_vv = nn.Parameter(torch.tensor(0.0))
        self.Z_ww = nn.Parameter(torch.tensor(0.0))
        self.K_pp = nn.Parameter(torch.tensor(0.0))
        self.M_qq = nn.Parameter(torch.tensor(0.0))
        self.N_rr = nn.Parameter(torch.tensor(0.0))

        #Sway-Yaw Coupling
        self.Y_rr = nn.Parameter(torch.tensor(0.0))
        self.N_vv = nn.Parameter(torch.tensor(0.0))

        # --------------------------
        # (6) Restoring / Buoyancy
        # --------------------------
        self.B   = nn.Parameter(torch.tensor(235.0))
        self.z_b = nn.Parameter(torch.tensor(-0.0))

        # --------------------------
        # (7) Constant Mass
        # --------------------------
        self.m_val = float(m_val)

        self.Crb = None
        self.Ca  = None
        self.C   = None
        self.D   = None

    # --- NEW helper ---
    def _get_inertias(self, device, dtype=torch.float32):
        """
        Returns (Ixx, Iyy, Izz) as tensors on the requested device/dtype.
        Uses learned params if enabled, otherwise CAD buffers.
        """
        if self.learn_inertia:
            Ixx = self.I_xx.to(device=device, dtype=dtype)
            Iyy = self.I_yy.to(device=device, dtype=dtype)
            Izz = self.I_zz.to(device=device, dtype=dtype)
        else:
            Ixx = self.Ixx_cad.to(device=device, dtype=dtype)
            Iyy = self.Iyy_cad.to(device=device, dtype=dtype)
            Izz = self.Izz_cad.to(device=device, dtype=dtype)
        return Ixx, Iyy, Izz

    # -------------------------------------------------------------------------
    # MASS MATRIX
    # -------------------------------------------------------------------------
    def build_mass_matrix(self) -> torch.Tensor:
        """
        M = M_RB + M_A (6x6)
        """
        device = self.X_dot_u.device
        dtype = self.X_dot_u.dtype

        Ma = torch.zeros((6, 6), dtype=dtype, device=device)

        # Diagonal added-mass terms
        Ma[0, 0] = -self.X_dot_u
        Ma[1, 1] = -self.Y_dot_v
        Ma[2, 2] = -self.Z_dot_w
        Ma[3, 3] = -self.K_dot_p
        Ma[4, 4] = -self.M_dot_q
        Ma[5, 5] = -self.N_dot_r

        # Symmetric sway-yaw added-mass coupling
        Ma[1, 5] = -self.Y_dot_r
        Ma[5, 1] = -self.Y_dot_r

        m  = torch.as_tensor(self.m_val, dtype=torch.float32, device=Ma.device)
        xg = self.x_g
        yg = self.y_g
        zg = self.z_g

        # NEW: get inertias (learned or CAD)
        Ixx, Iyy, Izz = self._get_inertias(device=Ma.device, dtype=torch.float32)

        Mrb = torch.zeros((6, 6), dtype=torch.float32, device=Ma.device)

        Mrb[0, 0] = m
        Mrb[1, 1] = m
        Mrb[2, 2] = m

        Mrb[0, 4] =  m * zg
        Mrb[0, 5] = -m * yg
        Mrb[1, 3] = -m * zg
        Mrb[1, 5] =  m * xg
        Mrb[2, 3] =  m * yg
        Mrb[2, 4] = -m * xg

        Mrb[3, 1] = -m * zg
        Mrb[3, 2] =  m * yg
        Mrb[3, 3] = Ixx

        Mrb[4, 0] =  m * zg
        Mrb[4, 2] = -m * xg
        Mrb[4, 4] = Iyy

        Mrb[5, 0] = -m * yg
        Mrb[5, 1] =  m * xg
        Mrb[5, 5] = Izz

        return Mrb + Ma

    # -------------------------------------------------------------------------
    # CORIOLIS: C = C_RB + C_A (with toggles)
    # -------------------------------------------------------------------------
    def build_coriolis(self, nu: torch.Tensor) -> torch.Tensor:
        """
        Returns C_total with flags controlling inclusion of Crb and Ca.
        nu: (B,6)
        """
        u, v, w, p, q, r = [nu[:, i] for i in range(6)]
        Bsz = nu.shape[0]

        m = torch.as_tensor(self.m_val, dtype=torch.float32, device=nu.device)

        Ixx, Iyy, Izz = self._get_inertias(device=nu.device, dtype=torch.float32)

        xg = self.x_g
        yg = self.y_g
        zg = self.z_g

        # --- Build Crb ---
        Crb = torch.zeros((Bsz, 6, 6), dtype=torch.float32, device=nu.device)

        Crb[:, 0, 1] = -m * r
        Crb[:, 0, 2] =  m * q
        Crb[:, 0, 3] =  m * (q * yg + r * zg)
        Crb[:, 0, 4] = -m * (q * xg)
        Crb[:, 0, 5] = -m * (r * xg)

        Crb[:, 1, 0] =  m * r
        Crb[:, 1, 2] = -m * p
        Crb[:, 1, 3] = -m * (p * yg)
        Crb[:, 1, 4] =  m * (p * xg + r * zg)
        Crb[:, 1, 5] = -m * (r * yg)

        Crb[:, 2, 0] = -m * q
        Crb[:, 2, 1] =  m * p
        Crb[:, 2, 3] = -m * (p * zg)
        Crb[:, 2, 4] = -m * (q * zg)
        Crb[:, 2, 5] =  m * (p * xg + q * yg)

        Crb[:, 3, 0] = -m * (q * yg + r * zg)
        Crb[:, 3, 1] =  m * (p * yg)
        Crb[:, 3, 2] =  m * (p * zg)
        Crb[:, 3, 4] =  Izz * r
        Crb[:, 3, 5] = -Iyy * q

        Crb[:, 4, 0] =  m * (q * xg)
        Crb[:, 4, 1] = -m * (p * xg + r * zg)
        Crb[:, 4, 2] =  m * (q * zg)
        Crb[:, 4, 3] = -Izz * r
        Crb[:, 4, 5] =  Ixx * p

        Crb[:, 5, 0] =  m * (r * xg)
        Crb[:, 5, 1] =  m * (r * yg)
        Crb[:, 5, 2] = -m * (p * xg + q * yg)
        Crb[:, 5, 3] =  Iyy * q
        Crb[:, 5, 4] = -Ixx * p

        # --- Build Ca (optional) ---
        Ca = torch.zeros_like(Crb)
        if self.use_Ca:
            a1 = self.X_dot_u * u
            a2 = self.Y_dot_v * v + self.Y_dot_r * r
            a3 = self.Z_dot_w * w

            b1 = self.K_dot_p * p
            b2 = self.M_dot_q * q
            b3 = self.Y_dot_r * v + self.N_dot_r * r #this appears like Y_dot_r because we are saying N_dot_v and Y_dot_r are equal to each other because of assuming symettry

            Ca[:, 0, 4] = -a3
            Ca[:, 0, 5] =  a2
            Ca[:, 1, 3] =  a3
            Ca[:, 1, 5] = -a1
            Ca[:, 2, 3] = -a2
            Ca[:, 2, 4] =  a1

            Ca[:, 3, 1] = -a3
            Ca[:, 3, 2] =  a2
            Ca[:, 3, 4] = -b3
            Ca[:, 3, 5] =  b2

            Ca[:, 4, 0] =  a3
            Ca[:, 4, 2] = -a1
            Ca[:, 4, 3] =  b3
            Ca[:, 4, 5] = -b1

            Ca[:, 5, 0] = -a2
            Ca[:, 5, 1] =  a1
            Ca[:, 5, 3] = -b2
            Ca[:, 5, 4] =  b1

        # --- Combine per flags ---
        C_total = torch.zeros_like(Crb)
        if self.use_Crb:
            C_total = C_total + Crb
        if self.use_Ca:
            C_total = C_total + Ca

        # Store for debug
        self.Crb = Crb
        self.Ca  = Ca
        self.C   = C_total

        return C_total

    # -------------------------------------------------------------------------
    # DAMPING
    # -------------------------------------------------------------------------
    def build_damping(self, nu: torch.Tensor) -> torch.Tensor:
        """
        Build D(nu) as a batch of 6x6 damping matrices.

        Current convention:
            tau_damping = D(nu) @ nu

        Diagonal damping:
            X from u
            Y from v
            Z from w
            K from p
            M from q
            N from r

        Sway-yaw off-diagonal damping:
            Y from r
            N from v
        """
        Bsz = nu.shape[0]
        device = nu.device
        dtype = nu.dtype

        u = nu[:, 0]
        v = nu[:, 1]
        w = nu[:, 2]
        p = nu[:, 3]
        q = nu[:, 4]
        r = nu[:, 5]

        D = torch.zeros((Bsz, 6, 6), dtype=dtype, device=device)

        # --------------------------
        # Diagonal damping terms
        # --------------------------
        D[:, 0, 0] = -1.0 * (self.X_u  + self.X_uu * torch.abs(u))
        D[:, 1, 1] = -1.0 * (self.Y_v  + self.Y_vv * torch.abs(v))
        D[:, 2, 2] = -1.0 * (self.Z_w  + self.Z_ww * torch.abs(w))
        D[:, 3, 3] = -1.0 * (self.K_p  + self.K_pp * torch.abs(p))
        D[:, 4, 4] = -1.0 * (self.M_q  + self.M_qq * torch.abs(q))
        D[:, 5, 5] = -1.0 * (self.N_r  + self.N_rr * torch.abs(r))

        # --------------------------
        # Off-diagonal sway-yaw damping
        # --------------------------
        # Y equation from yaw rate r:
        #   contribution = D[:,1,5] * r
        #                = -(Y_r + Y_rr*|r|) * r
        D[:, 1, 5] = -1.0 * (self.Y_r + self.Y_rr * torch.abs(r))

        # N equation from sway velocity v:
        #   contribution = D[:,5,1] * v
        #                = -(N_v + N_vv*|v|) * v
        D[:, 5, 1] = -1.0 * (self.N_v + self.N_vv * torch.abs(v))

        self.D = D
        return D

    # -------------------------------------------------------------------------
    # RESTORING FORCE
    # -------------------------------------------------------------------------
    def build_restoring_force(self, eta: torch.Tensor) -> torch.Tensor:
        """
        g(eta) in NED (z down)
        eta: (B,6) [x,y,z,phi,theta,psi]
        """
        phi   = eta[:, 3]
        theta = eta[:, 4]

        g0 = 9.8
        m = torch.as_tensor(self.m_val, dtype=torch.float32, device=eta.device)
        W = m * g0
        B = self.B.to(dtype=torch.float32, device=eta.device)

        x_G, y_G, z_G = self.x_g, self.y_g, self.z_g
        x_B = torch.zeros(1, device=eta.device)
        y_B = torch.zeros(1, device=eta.device)
        z_B = self.z_b.to(dtype=torch.float32, device=eta.device)

        WB    = W - B
        xW_xB = x_G * W - x_B * B
        yW_yB = y_G * W - y_B * B
        zW_zB = z_G * W - z_B * B

        tau_g = torch.stack([
            WB * torch.sin(theta),
            -WB * torch.cos(theta) * torch.sin(phi),
            -WB * torch.cos(theta) * torch.cos(phi),
            -yW_yB * torch.cos(theta) * torch.cos(phi) + zW_zB * torch.cos(theta) * torch.sin(phi),
            zW_zB * torch.sin(theta) + xW_xB * torch.cos(theta) * torch.cos(phi),
            -xW_xB * torch.cos(theta) * torch.sin(phi) - yW_yB * torch.sin(theta)
        ], dim=1)

        return tau_g

    # -------------------------------------------------------------------------
    # FORWARD (inverse dynamics prediction)
    # -------------------------------------------------------------------------
    def forward(self, nu_dot: torch.Tensor, nu: torch.Tensor, eta: torch.Tensor) -> torch.Tensor:
        """
        nu_dot: (B,6)
        nu:     (B,6)
        eta:    (B,6)
        returns tau: (B,6)
        """
        # Mass
        M = self.build_mass_matrix()                 # (6,6)
        tau_M = torch.matmul(nu_dot, M.T)            # (B,6)

        # Coriolis (optional pieces inside)
        C = self.build_coriolis(nu)                  # (B,6,6)
        tau_C = torch.einsum("bij,bj->bi", C, nu)    # (B,6)

        # Damping
        if self.use_D:
            D = self.build_damping(nu)               # (B,6,6)
            tau_D = torch.einsum("bij,bj->bi", D, nu)
        else:
            tau_D = torch.zeros_like(tau_M)

        # Restoring
        if self.use_g:
            tau_g = self.build_restoring_force(eta)  # (B,6)
        else:
            tau_g = torch.zeros_like(tau_M)

        return tau_M + tau_C + tau_D + tau_g

    # -------------------------------------------------------------------------
    # REGULARIZATION HELPERS (unchanged)
    # -------------------------------------------------------------------------
    def L2reg(self, selected_names, prior_means):
        loss = 0.0
        for name, param in self.named_parameters():
            if name in selected_names:
                mu = prior_means[name].to(param.device)
                loss += torch.sum((param - mu) ** 2)
        return loss

    def L1reg(self, param_names):
        l1reg_sum = 0.0
        for name, param in self.named_parameters():
            if name in param_names:
                l1reg_sum += torch.sum(torch.abs(param))
        return l1reg_sum


In [ ]:
model = ROVDynamicsModel()
optimizer = torch.optim.Adam(model.parameters(), lr=1.5)

n_epochs = 10000
verbose_option = True

ac_train  = nu_dot_train
v_train   = nu_train
eta_train_input = eta_train

for i in range(n_epochs):
    # Clear old gradients
    optimizer.zero_grad()

    # Forward pass: predict tau from measured nu_dot, nu, eta
    tau_pred = model(ac_train, v_train, eta_train_input)

    # MLE-style loss under fixed Gaussian noise assumption
    loss = F.mse_loss(tau_pred, y_train)

    # Backward pass: compute gradients
    loss.backward()

    # Update learnable parameters
    optimizer.step()

    # Print occasionally
    if verbose_option and (i % 100 == 0 or i == n_epochs - 1):
        print(f"Epoch {i:05d} | Loss: {loss.item():.6f}")

Epoch 0 | Loss: 9301.474609
Epoch 1 | Loss: 49339.277344
Epoch 2 | Loss: 11717.938477
Epoch 3 | Loss: 19218.011719
Epoch 4 | Loss: 32270.482422
Epoch 5 | Loss: 25096.337891
Epoch 6 | Loss: 12728.050781
Epoch 7 | Loss: 8966.575195
Epoch 8 | Loss: 14312.817383
Epoch 9 | Loss: 19331.009766
Epoch 10 | Loss: 17915.042969
Epoch 11 | Loss: 12523.515625
Epoch 12 | Loss: 8704.523438
Epoch 13 | Loss: 9159.768555
Epoch 14 | Loss: 11946.727539
Epoch 15 | Loss: 13361.729492
Epoch 16 | Loss: 11940.656250
Epoch 17 | Loss: 9266.501953
Epoch 18 | Loss: 7734.848633
Epoch 19 | Loss: 8159.738281
Epoch 20 | Loss: 9371.095703
Epoch 21 | Loss: 9758.210938
Epoch 22 | Loss: 8886.071289
Epoch 23 | Loss: 7633.699219
Epoch 24 | Loss: 7045.013672
Epoch 25 | Loss: 7288.693848
Epoch 26 | Loss: 7718.991699
Epoch 27 | Loss: 7691.032227
Epoch 28 | Loss: 7168.057617
Epoch 29 | Loss: 6602.263672
Epoch 30 | Loss: 6364.929199
Epoch 31 | Loss: 6415.809570
Epoch 32 | Loss: 6472.687012
Epoch 33 | Loss: 6346.926758
Epoch 34 | 

KeyboardInterrupt: 

In [19]:
print("\n=== Model Parameter Summary ===")
print(f"{'Parameter':<20} {'Value':>12}")
print("-" * 34)

for name, param in model.named_parameters():
    val = param.detach().cpu().numpy().item() if param.numel() == 1 else "array"
    print(f"{name:<20} {val:>12}")



=== Model Parameter Summary ===
Parameter                   Value
----------------------------------
X_dot_u              -32.12114334106445
Y_dot_v              24.674455642700195
Z_dot_w              34.812137603759766
K_dot_p              -1.6307796239852905
M_dot_q              1.939696192741394
N_dot_r              -0.28276053071022034
x_g                  0.001368940807878971
y_g                  4.740718213724904e-05
z_g                  0.01421545259654522
X_u                  -15.20694351196289
Y_v                  -39.89210891723633
Z_w                  -30.984771728515625
K_p                  0.027933182194828987
M_q                  6.749212265014648
N_r                  5.496082782745361
X_uu                 -43.795387268066406
Y_vv                 403.974365234375
Z_ww                 -165.18316650390625
K_pp                 0.07222075015306473
M_qq                 -4.840574741363525
N_rr                 -10.309252738952637
B                    233.21054077148438
z_b    

In [22]:
model.eval()
with torch.no_grad():
    tau_pred = model(nu_dot_test, nu_test, eta_test)

    # ======================================================
    # 1. Active DOF for this single-DOF file
    # ======================================================
    active_dof = "X"
    dof_labels = ["X", "Y", "Z", "K", "M", "N"]
    active_idx = dof_labels.index(active_dof)

    # ======================================================
    # 2. RMSE over ALL test samples
    #    This includes zero-force coast-down samples.
    # ======================================================
    error = tau_pred - y_test

    per_axis_rmse = torch.sqrt(torch.mean(error ** 2, dim=0))
    active_rmse = per_axis_rmse[active_idx]

    global_mse = torch.mean(error ** 2)
    global_rmse = torch.sqrt(global_mse)

    # ======================================================
    # 3. Normalization scales
    # ======================================================
    eps = 1e-6

    # Mean/max target magnitude over ALL samples
    # Includes X = 0 coast-down samples.
    mean_abs_target_all = torch.mean(torch.abs(y_test), dim=0)
    max_abs_target_all = torch.max(torch.abs(y_test), dim=0).values

    # Mean/max target magnitude over active-command samples only
    active_target = y_test[:, active_idx]
    active_command_mask = torch.abs(active_target) > 1e-3

    if torch.any(active_command_mask):
        mean_active_force = torch.mean(torch.abs(active_target[active_command_mask]))
        max_active_force = torch.max(torch.abs(active_target[active_command_mask]))
    else:
        mean_active_force = torch.tensor(float("nan"), device=y_test.device)
        max_active_force = torch.tensor(float("nan"), device=y_test.device)

    # Relative RMSE for active DOF
    rel_rmse_vs_all_mean = 100 * active_rmse / (mean_abs_target_all[active_idx] + eps)
    rel_rmse_vs_all_max = 100 * active_rmse / (max_abs_target_all[active_idx] + eps)

    rel_rmse_vs_active_mean = 100 * active_rmse / (mean_active_force + eps)
    rel_rmse_vs_active_max = 100 * active_rmse / (max_active_force + eps)

    # ======================================================
    # 4. Per-axis table
    # ======================================================
    rel_rmse_all_mean = 100 * per_axis_rmse / (mean_abs_target_all + eps)
    rel_rmse_all_max = 100 * per_axis_rmse / (max_abs_target_all + eps)

    # Avoid meaningless huge percentages for inactive zero-target axes
    inactive_axis_mask = max_abs_target_all < 1e-3
    rel_rmse_all_mean[inactive_axis_mask] = float("nan")
    rel_rmse_all_max[inactive_axis_mask] = float("nan")

    df = pd.DataFrame({
        "DOF": dof_labels,
        "RMSE": per_axis_rmse.cpu().numpy(),
        "Mean |Target| All": mean_abs_target_all.cpu().numpy(),
        "Max |Target| All": max_abs_target_all.cpu().numpy(),
        "Rel RMSE vs All Mean [%]": rel_rmse_all_mean.cpu().numpy(),
        "Rel RMSE vs All Max [%]": rel_rmse_all_max.cpu().numpy(),
    })

    # ======================================================
    # 5. Print results
    # ======================================================
    print("\n=== Test Set Error Metrics ===")
    print(f"Active DOF: {active_dof}")
    print(f"Global MSE over all samples/all axes : {global_mse.item():.6f}")
    print(f"Global RMSE over all samples/all axes: {global_rmse.item():.3f}")

    print(f"\n{active_dof} RMSE over ALL samples: {active_rmse.item():.3f}")

    print(f"\n{active_dof} relative RMSE normalizations:")
    print(f"  vs mean |{active_dof}| over ALL samples:            {rel_rmse_vs_all_mean.item():.3f}%")
    print(f"  vs max  |{active_dof}| over ALL samples:            {rel_rmse_vs_all_max.item():.3f}%")
    print(f"  vs mean |{active_dof}| over active-command samples: {rel_rmse_vs_active_mean.item():.3f}%")
    print(f"  vs max  |{active_dof}| over active-command samples: {rel_rmse_vs_active_max.item():.3f}%")

    print("\nPer-axis RMSE over ALL test samples:")
    print(df.to_string(index=False, float_format=lambda x: f"{x:8.3f}"))


=== Test Set Error Metrics ===
Active DOF: X
Global MSE over all samples/all axes : 19.639839
Global RMSE over all samples/all axes: 4.432

X RMSE over ALL samples: 10.589

X relative RMSE normalizations:
  vs mean |X| over ALL samples:            52.777%
  vs max  |X| over ALL samples:            13.665%
  vs mean |X| over active-command samples: 35.964%
  vs max  |X| over active-command samples: 13.665%

Per-axis RMSE over ALL test samples:
DOF     RMSE  Mean |Target| All  Max |Target| All  Rel RMSE vs All Mean [%]  Rel RMSE vs All Max [%]
  X   10.589             20.064            77.490                    52.777                   13.665
  Y    1.082              0.000             0.000                       NaN                      NaN
  Z    1.936              0.000             0.000                       NaN                      NaN
  K    0.056              0.000             0.000                       NaN                      NaN
  M    0.722              0.000             0.0

# Defender MAP Regression

In [16]:
# ==== PRIOR MEANS FROM TANK TESTING ===


prior_means = {
    "X_dot_u": torch.tensor(-33.61),
    "Y_dot_v": torch.tensor(-31.56),
    "Z_dot_w": torch.tensor(-79.58),
    "K_dot_p": torch.tensor(-0.10),
    "M_dot_q": torch.tensor(-0.46),
    "N_dot_r": torch.tensor(-0.70),

    "I_xx": torch.tensor(0.393), #from CAD
    "I_yy": torch.tensor(1.302), #from CAD
    "I_zz": torch.tensor(1.429), #from CAD

    "x_g": torch.tensor(0.0),
    "y_g": torch.tensor(0.0),
    "z_g": torch.tensor(0.0),

    "X_u": torch.tensor(-0.00),
    "Y_v": torch.tensor(-0.00),
    "Z_w": torch.tensor(-0.00),
    "K_p": torch.tensor(-0.00),
    "M_q": torch.tensor(-0.00),
    "N_r": torch.tensor(-0.00),

    "X_uu": torch.tensor(-42.49),
    "Y_vv": torch.tensor(-108.74),
    "Z_ww": torch.tensor(-128.31),
    "K_pp": torch.tensor(-0.08),
    "M_qq": torch.tensor(-1.61),
    "N_rr": torch.tensor(-2.69),

    "B": torch.tensor(236.00),
    "z_b": torch.tensor(-0.03)
}


In [18]:
# ============================================================
# === MAP configuration ======================================
# ============================================================

# Regularization strengths
l2_tight_coeff = 1e-2   # tight Gaussian prior on trusted single-DOF model parameters

# For this experiment, do not regularize new off-diagonal terms.
# We want the coupled maneuver data to learn them.
l2_loose_coeff = 0.0
l1_coeff       = 0.0

# Parameters from the trusted single-DOF prior model
L2_TIGHT = [
    # Diagonal added mass
    "X_dot_u", "Y_dot_v", "Z_dot_w",
    "K_dot_p", "M_dot_q", "N_dot_r",

    # CG offsets
    "x_g", "y_g", "z_g",

    # Diagonal linear damping
    "X_u", "Y_v", "Z_w",
    "K_p", "M_q", "N_r",

    # Diagonal quadratic damping
    "X_uu", "Y_vv", "Z_ww",
    "K_pp", "M_qq", "N_rr",

    # Restoring / buoyancy
    "B", "z_b",
]

# Only relevant if learn_inertia=True and these exist as trainable parameters
L2_TIGHT += ["I_xx", "I_yy", "I_zz"]

# Do not use loose/L1 regularization in this experiment
L2_LOOSE = []
L1_LINEAR = []


# ============================================================
# === Model + optimizer =====================================
# ============================================================

model = ROVDynamicsModel()
optimizer = optim.Adam(model.parameters(), lr=0.5)

device = next(model.parameters()).device

for k, v in prior_means.items():
    prior_means[k] = v.to(device).float()

model_param_names = {name for name, _ in model.named_parameters()}

# Keep only names that exist in both prior_means and model parameters
L2_TIGHT  = [p for p in L2_TIGHT  if p in prior_means and p in model_param_names]
L2_LOOSE  = [p for p in L2_LOOSE  if p in prior_means and p in model_param_names]
L1_LINEAR = [p for p in L1_LINEAR if p in model_param_names]

In [19]:
# ============================================================
# === MAP configuration ======================================
# ============================================================

n_epochs = 10000
verbose_option = True

# Tight regularization toward trusted single-DOF priors
l2_tight_coeff = 1e-2

# Off-diagonal terms are intentionally left unregularized
l2_loose_coeff = 0.0
l1_coeff = 0.0

# Trusted parameters from single-DOF prior model
L2_TIGHT = [
    # Diagonal added mass
    "X_dot_u", "Y_dot_v", "Z_dot_w",
    "K_dot_p", "M_dot_q", "N_dot_r",

    # CG offsets
    "x_g", "y_g", "z_g",

    # Diagonal linear damping
    "X_u", "Y_v", "Z_w",
    "K_p", "M_q", "N_r",

    # Diagonal quadratic damping
    "X_uu", "Y_vv", "Z_ww",
    "K_pp", "M_qq", "N_rr",

    # Restoring / buoyancy
    "B", "z_b",

    # Only included if learn_inertia=True
    "I_xx", "I_yy", "I_zz",
]

L2_LOOSE = []
L1_LINEAR = []

# ============================================================
# === Move priors to model device ============================
# ============================================================

device = next(model.parameters()).device

for k, v in prior_means.items():
    prior_means[k] = v.to(device).float()

model_param_names = {name for name, _ in model.named_parameters()}

# Keep only parameters that exist in both the model and prior_means
L2_TIGHT = [
    p for p in L2_TIGHT
    if p in prior_means and p in model_param_names
]

L2_LOOSE = [
    p for p in L2_LOOSE
    if p in prior_means and p in model_param_names
]

L1_LINEAR = [
    p for p in L1_LINEAR
    if p in model_param_names
]

# ============================================================
# === Initialize trusted parameters from prior means =========
# ============================================================

with torch.no_grad():
    for name, param in model.named_parameters():
        if name in prior_means:
            param.copy_(prior_means[name])

# ============================================================
# === Print regularization setup =============================
# ============================================================

regularized_params = set(L2_TIGHT + L2_LOOSE + L1_LINEAR)
unregularized_params = [
    name for name, _ in model.named_parameters()
    if name not in regularized_params
]

print("\n=== MAP regularization sets ===")
print(f"L2_TIGHT ({len(L2_TIGHT)}): {L2_TIGHT}")
print(f"L2_LOOSE ({len(L2_LOOSE)}): {L2_LOOSE}")
print(f"L1_LINEAR({len(L1_LINEAR)}): {L1_LINEAR}")

print("\nParameters NOT regularized:")
for name in unregularized_params:
    print(f"  - {name}")

print("================================\n")

# ============================================================
# === Training loop ==========================================
# ============================================================

for epoch in range(n_epochs):
    optimizer.zero_grad()

    tau_pred = model(nu_dot_train, nu_train, eta_train)

    # Gaussian NLL up to constant scale
    nll_loss = F.mse_loss(tau_pred, y_train)

    # Tight prior on trusted single-DOF parameters
    l2_tight_term = model.L2reg(L2_TIGHT, prior_means)

    # Empty by design in this experiment
    l2_loose_term = model.L2reg(L2_LOOSE, prior_means)
    l1_term = model.L1reg(L1_LINEAR)

    loss = (
        nll_loss
        + l2_tight_coeff * l2_tight_term
        + l2_loose_coeff * l2_loose_term
        + l1_coeff * l1_term
    )

    loss.backward()
    optimizer.step()

    if verbose_option and epoch % 50 == 0:
        print(
            f"Epoch {epoch:5d} | "
            f"Loss={loss.item():.6f} | "
            f"NLL={nll_loss.item():.6f} | "
            f"L2tight={l2_tight_term.item():.6f}"
        )


=== MAP regularization sets ===
L2_TIGHT (23): ['X_dot_u', 'Y_dot_v', 'Z_dot_w', 'K_dot_p', 'M_dot_q', 'N_dot_r', 'x_g', 'y_g', 'z_g', 'X_u', 'Y_v', 'Z_w', 'K_p', 'M_q', 'N_r', 'X_uu', 'Y_vv', 'Z_ww', 'K_pp', 'M_qq', 'N_rr', 'B', 'z_b']
L2_LOOSE (0): []
L1_LINEAR(0): []

Parameters NOT regularized:
  - Y_dot_r
  - Y_r
  - N_v
  - Y_rr
  - N_vv

Epoch     0 | Loss=85.506531 | NLL=85.506531 | L2tight=0.000000
Epoch    50 | Loss=42.564075 | NLL=34.772678 | L2tight=779.139893
Epoch   100 | Loss=24.442781 | NLL=18.275141 | L2tight=616.764099
Epoch   150 | Loss=23.978588 | NLL=18.075430 | L2tight=590.315857
Epoch   200 | Loss=23.701487 | NLL=17.945761 | L2tight=575.572632
Epoch   250 | Loss=23.448059 | NLL=17.817043 | L2tight=563.101501
Epoch   300 | Loss=23.226856 | NLL=17.717108 | L2tight=550.974792
Epoch   350 | Loss=23.041809 | NLL=17.647282 | L2tight=539.452698
Epoch   400 | Loss=22.892735 | NLL=17.604433 | L2tight=528.830200
Epoch   450 | Loss=22.776674 | NLL=17.584126 | L2tight=519.2

KeyboardInterrupt: 

In [20]:
# ============================================================
# === Print learned MAP parameters ===========================
# ============================================================

param_groups = {
    "Added Mass": [
        "X_dot_u", "Y_dot_v", "Z_dot_w",
        "K_dot_p", "M_dot_q", "N_dot_r",
        "Y_dot_r",
    ],
    "CG Offsets": [
        "x_g", "y_g", "z_g",
    ],
    "Linear Damping": [
        "X_u", "Y_v", "Z_w",
        "K_p", "M_q", "N_r",
        "Y_r", "N_v",
    ],
    "Quadratic Damping": [
        "X_uu", "Y_vv", "Z_ww",
        "K_pp", "M_qq", "N_rr",
        "Y_rr", "N_vv",
    ],
    "Restoring / Buoyancy": [
        "B", "z_b",
    ],
}

model_params = dict(model.named_parameters())

print("\n=== Learned MAP Parameters ===")

for group_name, names in param_groups.items():
    print(f"\n--- {group_name} ---")
    for name in names:
        if name in model_params:
            value = model_params[name].detach().cpu().item()
            print(f"{name:12s} = {value: .6f}")


=== Learned MAP Parameters ===

--- Added Mass ---
X_dot_u      = -25.380404
Y_dot_v      = -20.288538
Z_dot_w      = -74.981529
K_dot_p      =  0.386802
M_dot_q      =  1.276141
N_dot_r      =  1.322868
Y_dot_r      =  0.137094

--- CG Offsets ---
x_g          = -0.000525
y_g          = -0.000511
z_g          =  0.000153

--- Linear Damping ---
X_u          = -13.198017
Y_v          = -5.373650
Z_w          = -0.213400
K_p          = -0.011144
M_q          = -0.075601
N_r          = -1.512524
Y_r          =  44.992886
N_v          = -1.176321

--- Quadratic Damping ---
X_uu         = -48.786266
Y_vv         = -109.836525
Z_ww         = -128.315018
K_pp         = -0.081463
M_qq         = -1.615900
N_rr         = -3.504406
Y_rr         = -33.007469
N_vv         = -24.731926

--- Restoring / Buoyancy ---
B            =  236.681610
z_b          = -0.004623
